# PRISM — Data Exploration Notebook

**Purpose:** Validate data fetching, feature engineering, and ICC detection on real/synthetic data.

**Sections:**
1. Load EUR/USD H1 data (yfinance placeholder)
2. Plot price + volume
3. Compute all technical features
4. Feature correlation matrix
5. ICC detection demo with annotations
6. FRED macro data alignment


In [ ]:
import sys
from pathlib import Path

# Add repo root to path
ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

plt.style.use('dark_background')
plt.rcParams['figure.figsize'] = (16, 6)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('Setup complete')

## 1. Load EUR/USD H1 Data

Uses `yfinance` as a placeholder when no Tiingo API key is available.
Switch to `prism.data.tiingo.get_intraday` for production data.

In [ ]:
import os

START_DATE = '2024-01-01'
END_DATE   = '2026-04-01'
INSTRUMENT = 'EURUSD'

if os.environ.get('TIINGO_API_KEY'):
    print('Tiingo key found — using production data')
    from prism.data.tiingo import get_intraday
    df_raw = get_intraday(INSTRUMENT, START_DATE, END_DATE, freq='1hour')
    df_raw = df_raw.set_index('datetime')
else:
    print('No Tiingo key — falling back to yfinance (EURUSD=X)')
    import yfinance as yf
    ticker = yf.Ticker('EURUSD=X')
    df_raw = ticker.history(start=START_DATE, end=END_DATE, interval='1h')
    df_raw = df_raw.rename(columns={
        'Open': 'open', 'High': 'high',
        'Low': 'low', 'Close': 'close', 'Volume': 'volume'
    })
    df_raw.index = df_raw.index.tz_convert('UTC')

df_raw = df_raw[['open', 'high', 'low', 'close', 'volume']].dropna()
print(f'Loaded {len(df_raw):,} rows  |  {df_raw.index[0]} → {df_raw.index[-1]}')
df_raw.tail(3)

## 2. Price + Volume Chart

In [ ]:
# Use last 500 bars for legibility
df_plot = df_raw.tail(500).copy()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(16, 8), gridspec_kw={'height_ratios': [3, 1]}, sharex=True)

ax1.plot(df_plot.index, df_plot['close'], color='#00d4aa', linewidth=0.8, label='Close')
ax1.fill_between(df_plot.index, df_plot['low'], df_plot['high'], alpha=0.15, color='#00d4aa', label='H-L range')
ax1.set_title(f'{INSTRUMENT} — H1 Price (last 500 bars)', fontsize=14)
ax1.set_ylabel('Price')
ax1.legend(loc='upper left')

ax2.bar(df_plot.index, df_plot['volume'], color='#4a9eff', alpha=0.7, width=0.04)
ax2.set_ylabel('Volume')
ax2.set_xlabel('Date (UTC)')

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 3. Technical Feature Engineering

In [ ]:
from prism.data.pipeline import (
    _ema, _atr, _rsi, _macd, _stochastic, _bollinger_pct, _obv, _classify_session
)

df = df_raw.copy()
c, h, l, v = df['close'], df['high'], df['low'], df['volume']

# Log returns
log_c = np.log(c)
df['return_1']  = log_c.diff(1)
df['return_5']  = log_c.diff(5)
df['return_20'] = log_c.diff(20)
df['return_50'] = log_c.diff(50)

# Volatility
df['atr_14'] = _atr(h, l, c, 14)
df['atr_50'] = _atr(h, l, c, 50)
df['hv_20']  = log_c.diff().rolling(20).std() * np.sqrt(252 * 24)

# Trend
df['ema_9']        = _ema(c, 9)
df['ema_21']       = _ema(c, 21)
df['ema_50']       = _ema(c, 50)
df['ema_200']      = _ema(c, 200)
df['ema_9_slope']  = df['ema_9'].diff()
df['ema_21_slope'] = df['ema_21'].diff()

# Momentum
df['rsi_14'] = _rsi(c, 14)
df['macd'], df['macd_signal'], df['macd_hist'] = _macd(c)
df['stoch_k'], df['stoch_d'] = _stochastic(h, l, c)

# Volume
df['obv_change'] = _obv(c, v).diff()

# Bollinger
df['bb_pct'] = _bollinger_pct(c)

# Calendar
df['session'] = df.index.hour.map(_classify_session)
df['day_of_week'] = df.index.dayofweek

df_feat = df.dropna()
print(f'Feature matrix: {df_feat.shape[0]:,} rows × {df_feat.shape[1]} cols')
df_feat[['close', 'rsi_14', 'macd', 'bb_pct', 'atr_14']].tail(5)

In [ ]:
# Plot key indicators
sample = df_feat.tail(300)

fig, axes = plt.subplots(4, 1, figsize=(16, 14), sharex=True)

# Price + EMAs
ax = axes[0]
ax.plot(sample.index, sample['close'],   color='white',   linewidth=0.7, label='Close')
ax.plot(sample.index, sample['ema_21'],  color='#FFD700', linewidth=1.0, label='EMA 21')
ax.plot(sample.index, sample['ema_50'],  color='#FF6B35', linewidth=1.0, label='EMA 50')
ax.plot(sample.index, sample['ema_200'], color='#4a9eff', linewidth=1.0, label='EMA 200')
ax.set_title('Price + EMAs')
ax.legend(loc='upper left', fontsize=8)

# RSI
ax = axes[1]
ax.plot(sample.index, sample['rsi_14'], color='#00d4aa', linewidth=0.8)
ax.axhline(70, color='red',   linestyle='--', alpha=0.5, label='Overbought')
ax.axhline(30, color='green', linestyle='--', alpha=0.5, label='Oversold')
ax.set_ylim(0, 100)
ax.set_title('RSI 14')
ax.legend(loc='upper left', fontsize=8)

# MACD
ax = axes[2]
ax.plot(sample.index, sample['macd'],        color='#4a9eff', linewidth=0.8, label='MACD')
ax.plot(sample.index, sample['macd_signal'], color='#FF6B35', linewidth=0.8, label='Signal')
hist_colors = ['#00d4aa' if x >= 0 else '#ff4444' for x in sample['macd_hist']]
ax.bar(sample.index, sample['macd_hist'], color=hist_colors, alpha=0.6, width=0.04, label='Histogram')
ax.set_title('MACD (12, 26, 9)')
ax.legend(loc='upper left', fontsize=8)

# Stochastic
ax = axes[3]
ax.plot(sample.index, sample['stoch_k'], color='#00d4aa', linewidth=0.8, label='%K')
ax.plot(sample.index, sample['stoch_d'], color='#FF6B35', linewidth=0.8, label='%D')
ax.axhline(80, color='red',   linestyle='--', alpha=0.5)
ax.axhline(20, color='green', linestyle='--', alpha=0.5)
ax.set_ylim(0, 100)
ax.set_title('Stochastic (14, 3)')
ax.legend(loc='upper left', fontsize=8)

fig.autofmt_xdate()
plt.tight_layout()
plt.show()

## 4. Feature Correlation Matrix

In [ ]:
feature_cols = [
    'return_1', 'return_5', 'return_20',
    'atr_14', 'hv_20',
    'ema_9_slope', 'ema_21_slope',
    'rsi_14', 'macd_hist', 'stoch_k',
    'bb_pct', 'obv_change',
    'day_of_week', 'session',
]

corr = df_feat[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f',
    cmap='RdYlGn', center=0, vmin=-1, vmax=1,
    linewidths=0.5, ax=ax, annot_kws={'size': 8},
)
ax.set_title('PRISM Feature Correlation Matrix', fontsize=14, pad=12)
plt.tight_layout()
plt.show()

# Print highly correlated pairs
print('\nHighly correlated pairs (|r| > 0.7):')
for i in range(len(corr.columns)):
    for j in range(i + 1, len(corr.columns)):
        if abs(corr.iloc[i, j]) > 0.7:
            print(f'  {corr.columns[i]:20s} ↔ {corr.columns[j]:20s}  r={corr.iloc[i, j]:.3f}')

## 5. ICC Pattern Detection — Annotated Chart

In [ ]:
from prism.signal.icc import detect_swing_points, get_icc_entry

# Use last 200 bars for ICC detection demo
df_icc = df_raw.tail(200).copy()
df_icc = detect_swing_points(df_icc, lookback=5)

swing_highs = df_icc[df_icc['swing_high']]
swing_lows  = df_icc[df_icc['swing_low']]

# Attempt ICC entry detection
icc_signal = get_icc_entry(df_icc, pip_value=0.0001)

fig, ax = plt.subplots(figsize=(16, 7))

ax.plot(df_icc.index, df_icc['close'], color='white', linewidth=0.8, label='Close', zorder=2)

# Swing highs and lows
ax.scatter(swing_highs.index, swing_highs['high'] + 0.0002,
           marker='v', color='#ff4444', s=60, zorder=5, label='Swing High')
ax.scatter(swing_lows.index, swing_lows['low'] - 0.0002,
           marker='^', color='#00d4aa', s=60, zorder=5, label='Swing Low')

# ICC signal annotation
if icc_signal:
    entry = icc_signal['entry']
    sl    = icc_signal['sl']
    phase = icc_signal['phase']
    ax.axhline(entry, color='#FFD700', linestyle='--', linewidth=1.5,
               label=f'ICC Entry ({phase}): {entry:.5f}')
    ax.axhline(sl, color='#ff4444', linestyle=':', linewidth=1.5,
               label=f'ICC SL: {sl:.5f}')
    ax.annotate(
        f'{phase}\nEntry: {entry:.5f}\nSL: {sl:.5f}\nCorrection: {icc_signal["correction_pct"]:.1f}%',
        xy=(df_icc.index[-1], entry),
        xytext=(df_icc.index[-30], entry + 0.003),
        color='#FFD700', fontsize=9,
        arrowprops=dict(arrowstyle='->', color='#FFD700'),
    )
else:
    ax.set_title(f'{INSTRUMENT} H1 — ICC Swing Point Detection (no active setup)', fontsize=13)

ax.set_title(
    f'{INSTRUMENT} H1 — ICC Detection | Swing Highs/Lows | Last 200 bars',
    fontsize=13,
)
ax.set_ylabel('Price')
ax.legend(loc='upper left', fontsize=9)
fig.autofmt_xdate()
plt.tight_layout()
plt.show()

print(f'Swing Highs: {len(swing_highs)} | Swing Lows: {len(swing_lows)}')
print(f'ICC Signal: {icc_signal}')

## 6. FRED Macro Data Alignment

In [ ]:
if os.environ.get('FRED_API_KEY'):
    print('FRED key found — fetching macro data')
    from prism.data.fred import get_macro_features
    macro = get_macro_features(START_DATE, END_DATE)
    macro['date'] = pd.to_datetime(macro['date'])
    macro = macro.set_index('date')

    # Align with price data
    price_daily = df_raw['close'].resample('D').last()
    aligned = pd.concat([price_daily.rename('eurusd_close'), macro], axis=1).ffill()

    fig, axes = plt.subplots(3, 1, figsize=(16, 12), sharex=True)

    axes[0].plot(aligned.index, aligned['eurusd_close'], color='white', linewidth=0.8)
    axes[0].set_title('EUR/USD Close (daily)')
    axes[0].set_ylabel('Price')

    axes[1].plot(aligned.index, aligned.get('yield_10y', pd.Series()),
                 color='#FFD700', label='10Y Yield')
    axes[1].plot(aligned.index, aligned.get('yield_2y', pd.Series()),
                 color='#FF6B35', label='2Y Yield')
    if 'yield_spread' in aligned.columns:
        axes[1].plot(aligned.index, aligned['yield_spread'],
                     color='#4a9eff', linestyle='--', label='Spread (10Y-2Y)')
    axes[1].set_title('US Treasury Yields')
    axes[1].set_ylabel('Yield (%)')
    axes[1].legend(loc='upper left', fontsize=8)

    axes[2].plot(aligned.index, aligned.get('vix', pd.Series()),
                 color='#ff4444', linewidth=0.8, label='VIX')
    axes[2].axhline(20, color='orange', linestyle='--', alpha=0.5, label='VIX 20')
    axes[2].axhline(30, color='red', linestyle='--', alpha=0.5, label='VIX 30')
    axes[2].set_title('VIX Fear Index')
    axes[2].set_ylabel('VIX')
    axes[2].legend(loc='upper left', fontsize=8)

    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()

    print('\nMacro snapshot (latest):')
    print(macro.tail(1).T)
else:
    print('FRED_API_KEY not set — skipping macro data section')
    print('To enable: export FRED_API_KEY=your_key')
    print('Free key: https://fred.stlouisfed.org/docs/api/api_key.html')

---
## Summary

| Module | Status |
|--------|--------|
| Price data (yfinance/Tiingo) | ✅ Loaded |
| Technical features | ✅ Computed |
| Correlation matrix | ✅ Visualised |
| ICC detection | ✅ Running |
| FRED macro | Requires FRED_API_KEY |

**Next steps:**
- Set `TIINGO_API_KEY` and rerun with real intraday data
- Set `FRED_API_KEY` for macro features
- Run `scripts/download_historical_data.py` to cache full 2018–2026 dataset
- Proceed to `02_model_training.ipynb` for ML layer training